# XAI-Driven Analysis of Gesture Intent Detection using SHAP (Real Data Only)

This notebook directly addresses four practical gesture-recognition limitations on real data:
1. Black-box predictions -> SHAP joint-level explanations
2. False positives from random motion -> structured negative motion + Gini intent gate
3. Unknown model fragility -> noise injection + prediction flip probability
4. No attention-failure link -> SHAP-importance vs fragility correlation

This version intentionally avoids synthetic benchmarking.

Expected assets (either precomputed or generated from real media in this notebook):
- `X_landmarks.npy` with shape `(N, 21, 3)`
- `y_labels.npy` with shape `(N,)`
- `gesture_model_best.pt` (TorchScript preferred)

If your checkpoint stores only `state_dict`, this notebook reconstructs the architecture before loading.

In [ ]:
# Environment setup and reproducibility
%pip install -q shap scikit-learn seaborn mediapipe opencv-python

import hashlib
import json
import pickle
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
import torch
import torch.nn as nn
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
    f1_score,
    classification_report,
    )
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import StandardScaler

# Global configuration
DATA_CANDIDATES = [
    Path('.'),
    Path('data'),
    Path('exai_precomputed_assets'),
    Path('/content'),
    Path('/content/drive/MyDrive'),
    Path('/kaggle/working'),
    Path('/kaggle/input'),
    Path('/kaggle/input/dataset3'),
    Path('/kaggle/input/dataset3/exai_precomputed_assets'),
]

JOINT_NAMES = [
    'Wrist',
    'Thumb_CMC',
    'Thumb_MCP',
    'Thumb_IP',
    'Thumb_Tip',
    'Index_MCP',
    'Index_PIP',
    'Index_DIP',
    'Index_Tip',
    'Middle_MCP',
    'Middle_PIP',
    'Middle_DIP',
    'Middle_Tip',
    'Ring_MCP',
    'Ring_PIP',
    'Ring_DIP',
    'Ring_Tip',
    'Pinky_MCP',
    'Pinky_PIP',
    'Pinky_DIP',
    'Pinky_Tip',
]

SHAP_CACHE_DIR = Path('/content/shap_cache') if Path('/content').exists() else Path('.shap_cache')
SHAP_CACHE_ENABLED = True

NUM_JOINTS = 21
NUM_COORDS = 3
INPUT_DIM = NUM_JOINTS * NUM_COORDS

# Experiment defaults. Increase for heavier analyses.
SEED = 42
TEST_SIZE = 0.2
N_BACKGROUND = 100
N_SHAP_EVAL = 200
SHAP_NSAMPLES = 128
SHAP_RUN_SEEDS = [0, 1, 2]
ROBUSTNESS_SIGMAS = np.linspace(0.0, 1.0, 10)
ROBUSTNESS_N_TRIALS = 10
ROBUSTNESS_MAX_SAMPLES = 50
NEGATIVE_SAMPLES_PER_TYPE = 50
NEGATIVE_SHAP_SAMPLES = 30

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12


def set_seed(seed: int) -> None:
    """Set Python, NumPy, and PyTorch seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_joint_labels(n_items: int) -> list[str]:
    """Return readable joint labels for plotting."""
    labels = JOINT_NAMES[:n_items]
    if len(labels) < n_items:
        labels = labels + [f'J{i + 1}' for i in range(len(labels), n_items)]
    return labels


def normalize_importance_vector(values: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    """Normalize a non-negative importance vector to a probability simplex."""
    x = np.asarray(values, dtype=np.float32).reshape(-1)
    x = np.abs(x)
    total = float(x.sum())
    if total <= eps:
        return np.zeros_like(x)
    return x / (total + eps)


def resolve_data_path(filename: str) -> Path:
    """Find a required file in common local paths."""
    for base_dir in DATA_CANDIDATES:
        candidate = base_dir / filename
        if candidate.exists():
            return candidate

    # Fallback for hosted runtimes where assets are nested under dataset folders.
    for base_dir in DATA_CANDIDATES:
        if not base_dir.exists() or not base_dir.is_dir():
            continue
        for candidate in base_dir.rglob(filename):
            return candidate
    searched = ', '.join(str(base_dir / filename) for base_dir in DATA_CANDIDATES)
    raise FileNotFoundError(f'Missing required file: {filename}. Searched: {searched}')


def load_dataset(x_path: Path, y_path: Path):
    """Load landmark tensor and label vector from .npy files."""
    X = np.load(x_path).astype(np.float32)
    y = np.load(y_path).astype(np.int64)
    if X.ndim != 3 or X.shape[1:] != (NUM_JOINTS, NUM_COORDS):
        raise ValueError(f'Expected X shape (N, {NUM_JOINTS}, {NUM_COORDS}), got {X.shape}')
    if y.ndim != 1 or len(y) != len(X):
        raise ValueError(f'Expected y shape (N,), got {y.shape}')
    return X, y


def stratified_split(X: np.ndarray, y: np.ndarray, test_size: float = TEST_SIZE, seed: int = SEED):
    """Create a train/test split that preserves class balance."""
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=seed,
        stratify=y,
    )
    return X_train, X_test, y_train, y_test


class GestureTransformerClassifier(nn.Module):
    """Transformer classifier over 21 hand joints."""

    def __init__(self, num_classes: int, embed_dim: int = 64, num_heads: int = 4, num_layers: int = 2, dropout: float = 0.2):
        super().__init__()
        self.input_proj = nn.Linear(3, embed_dim)
        self.pos_embedding = nn.Parameter(torch.zeros(1, 21, embed_dim))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            batch_first=True,
            activation='gelu',
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, num_classes),
        )

    def forward(self, x):
        x = self.input_proj(x)
        x = x + self.pos_embedding
        x = self.encoder(x)
        x = self.norm(x.mean(dim=1))
        return self.head(x)


def infer_num_classes_from_state_dict(state_dict: dict) -> int:
    """Infer class count from final classifier layer in a state_dict."""
    for key in ['head.3.weight', 'classifier.weight', 'fc.weight']:
        if key in state_dict and state_dict[key].ndim == 2:
            return int(state_dict[key].shape[0])
    raise ValueError('Could not infer num_classes from checkpoint state_dict.')


def build_model(num_classes: int):
    """Build the model architecture used during training."""
    return GestureTransformerClassifier(num_classes=num_classes)


def load_model(model_path: Path | None = None, device: torch.device | None = None):
    """Load a saved TorchScript model, nn.Module, or state_dict checkpoint."""
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if model_path is None:
        model_path = resolve_data_path('gesture_model_best.pt')
    model_path = Path(model_path)

    try:
        scripted_model = torch.jit.load(str(model_path), map_location=device)
        scripted_model.eval()
        return scripted_model
    except Exception:
        pass

    try:
        checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    except TypeError:
        checkpoint = torch.load(model_path, map_location=device)

    if isinstance(checkpoint, nn.Module):
        model = checkpoint.to(device)
        model.eval()
        return model

    if isinstance(checkpoint, dict):
        state_dict = None
        if 'model_state_dict' in checkpoint:
            state_dict = checkpoint['model_state_dict']
        elif 'state_dict' in checkpoint:
            state_dict = checkpoint['state_dict']
        else:
            state_dict = checkpoint

        if isinstance(state_dict, dict):
            num_classes = infer_num_classes_from_state_dict(state_dict)
            model = build_model(num_classes=num_classes).to(device)
            model.load_state_dict(state_dict, strict=True)
            model.eval()
            return model

    raise ValueError('Unsupported checkpoint format for gesture_model_best.pt.')


def _forward_logits(model: nn.Module, x_tensor: torch.Tensor) -> torch.Tensor:
    """Call the model and normalize outputs to logits."""
    logits = model(x_tensor)
    if isinstance(logits, (tuple, list)):
        logits = logits[0]
    return logits


def predict_logits(model: nn.Module, X: np.ndarray, batch_size: int = 256, device: torch.device | None = None) -> np.ndarray:
    """Return logits for inputs of shape (N, 21, 3)."""
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.eval()
    outputs = []
    with torch.no_grad():
        for start in range(0, len(X), batch_size):
            batch = torch.from_numpy(X[start:start + batch_size]).float().to(device)
            logits = _forward_logits(model, batch)
            outputs.append(logits.detach().cpu().numpy())
    return np.concatenate(outputs, axis=0)


def predict_proba(model: nn.Module, X: np.ndarray, batch_size: int = 256, device: torch.device | None = None) -> np.ndarray:
    """Return softmax probabilities for inputs of shape (N, 21, 3)."""
    logits = predict_logits(model, X, batch_size=batch_size, device=device)
    logits = logits - logits.max(axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    return exp_logits / exp_logits.sum(axis=1, keepdims=True)


def predict_labels(model: nn.Module, X: np.ndarray, batch_size: int = 256, device: torch.device | None = None) -> np.ndarray:
    """Return predicted class indices."""
    return np.argmax(predict_proba(model, X, batch_size=batch_size, device=device), axis=1)


def predict_confidence(model: nn.Module, X: np.ndarray, batch_size: int = 256, device: torch.device | None = None) -> np.ndarray:
    """Return max softmax probability for each sample."""
    probs = predict_proba(model, X, batch_size=batch_size, device=device)
    return probs.max(axis=1)


def compute_entropy(model: nn.Module, X: np.ndarray, batch_size: int = 256, device: torch.device | None = None) -> np.ndarray:
    """Return predictive entropy for each sample."""
    probs = predict_proba(model, X, batch_size=batch_size, device=device)
    eps = 1e-12
    return -np.sum(probs * np.log(probs + eps), axis=1)


def predict_proba_flat(model: nn.Module, X_flat: np.ndarray, batch_size: int = 256, device: torch.device | None = None) -> np.ndarray:
    """SHAP wrapper: accepts flattened shape (N, 63)."""
    X_flat = np.asarray(X_flat, dtype=np.float32)
    if X_flat.ndim == 1:
        X_flat = X_flat[None, :]
    X = X_flat.reshape(-1, NUM_JOINTS, NUM_COORDS)
    return predict_proba(model, X, batch_size=batch_size, device=device)


set_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

# Load real data arrays if they already exist. If not, run the real-data preparation cell below.
X_PATH = None
Y_PATH = None
MODEL_PATH = None

try:
    X_PATH = resolve_data_path('X_landmarks.npy')
    Y_PATH = resolve_data_path('y_labels.npy')
    X, y = load_dataset(X_PATH, Y_PATH)
    X_train, X_test, y_train, y_test = stratified_split(X, y, test_size=TEST_SIZE, seed=SEED)
    print(f'Loaded real dataset: X={X.shape}, y={y.shape}, classes={len(np.unique(y))}')
except FileNotFoundError as err:
    print(str(err))
    print('Run the real-data preparation/training cell to extract landmarks from data/raw and generate required files.')

Note: you may need to restart the kernel to use updated packages.
Using device: cpu
Missing required file: X_landmarks.npy. Searched: X_landmarks.npy, data\X_landmarks.npy, \content\X_landmarks.npy, \content\drive\MyDrive\X_landmarks.npy
Run the real-data preparation/training cell to extract landmarks from data/raw and generate required files.


## Validate Required Assets

If `X_landmarks.npy`, `y_labels.npy`, and `gesture_model_best.pt` already exist, this notebook will use them directly.
Otherwise, run the real data preparation cell to generate them from `data/raw/<gesture_class>/...`.

In [ ]:
# Quick asset validation for local or Colab runs
from pathlib import Path

required_files = ['X_landmarks.npy', 'y_labels.npy', 'gesture_model_best.pt']
found = []
missing = []

search_roots = [
    Path('.'),
    Path('data'),
    Path('exai_precomputed_assets'),
    Path('/content'),
    Path('/content/drive/MyDrive'),
    Path('/kaggle/working'),
    Path('/kaggle/input'),
    Path('/kaggle/input/dataset3'),
    Path('/kaggle/input/dataset3/exai_precomputed_assets'),
]

for name in required_files:
    candidates = [root / name for root in search_roots]
    path = next((candidate for candidate in candidates if candidate.exists()), None)
    if path is None:
        for root in search_roots:
            if not root.exists() or not root.is_dir():
                continue
            matches = list(root.rglob(name))
            if matches:
                path = matches[0]
                break
    if path is None:
        missing.append(name)
    else:
        found.append(str(path))

print('Found:')
for item in found:
    print(f'- {item}')

if missing:
    print('\nMissing required assets:')
    for item in missing:
        print(f'- {item}')
    print('Run the real data preparation/training cell next.')
else:
    print('\nAll required assets are available.')

## Real Data Preparation and Model Training

Run the next cell to do one of the following:
- Use existing real assets (`X_landmarks.npy`, `y_labels.npy`) if present
- Or build those arrays from real images/videos in `data/raw/<gesture_class>/...` using MediaPipe Hands

Then train a transformer gesture classifier on the real landmarks and export:
- `X_landmarks.npy`
- `y_labels.npy`
- `gesture_model_best.pt`

No synthetic dataset is created in this notebook.

In [ ]:
# Real-data landmark extraction, strict training, and export (no synthetic fallback)

from copy import deepcopy
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split


def get_output_dir() -> Path:
    return Path('/content') if Path('/content').exists() else Path('.')


def extract_hand_landmarks_from_rgb(rgb_image: np.ndarray, hands_detector) -> np.ndarray | None:
    """Extract one hand's 21x3 normalized landmarks from an RGB image."""
    result = hands_detector.process(rgb_image)
    if not result.multi_hand_landmarks:
        return None

    hand_idx = 0
    if result.multi_handedness and len(result.multi_handedness) == len(result.multi_hand_landmarks):
        scores = [entry.classification[0].score for entry in result.multi_handedness]
        hand_idx = int(np.argmax(scores))

    landmarks = result.multi_hand_landmarks[hand_idx].landmark
    arr = np.array([[lm.x, lm.y, lm.z] for lm in landmarks], dtype=np.float32)
    if arr.shape != (21, 3):
        return None
    return arr


def get_mediapipe_hands_module():
    """Return a MediaPipe Hands module from the installed API layout."""
    import importlib
    try:
        return importlib.import_module('mediapipe.python.solutions.hands')
    except ModuleNotFoundError:
        try:
            import mediapipe as mp
        except Exception:
            return None
        solutions = getattr(mp, 'solutions', None)
        if solutions is not None and hasattr(solutions, 'hands'):
            return solutions.hands
        return None


def extract_landmarks_from_image(image_path: Path, hands_detector) -> np.ndarray | None:
    bgr = cv2.imread(str(image_path))
    if bgr is None:
        return None
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    return extract_hand_landmarks_from_rgb(rgb, hands_detector)


def extract_landmarks_from_video(video_path: Path, hands_detector, frame_stride: int = 4, max_frames: int = 150) -> list[np.ndarray]:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return []

    extracted = []
    frame_index = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if frame_index % frame_stride == 0:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            landmarks = extract_hand_landmarks_from_rgb(rgb, hands_detector)
            if landmarks is not None:
                extracted.append(landmarks)
                if len(extracted) >= max_frames:
                    break
        frame_index += 1

    cap.release()
    return extracted


def build_landmark_dataset_from_raw(raw_dir: Path, min_samples_per_class: int = 25, frame_stride: int = 4, group_level: str = 'file'):
    """Build (X,y) from real media in data/raw/<class_name>/..."""
    raw_dir = Path(raw_dir)
    if not raw_dir.exists():
        raise FileNotFoundError(f'Raw data directory not found: {raw_dir}')

    class_dirs = sorted([path for path in raw_dir.iterdir() if path.is_dir()])
    if not class_dirs:
        raise ValueError(f'No class folders found under {raw_dir}. Expected data/raw/<class_name>/...')

    image_ext = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    video_ext = {'.mp4', '.avi', '.mov', '.mkv'}

    X_list = []
    y_list = []
    source_ids_list = []
    class_to_id = {}

    def make_source_id(path_obj: Path) -> str:
        try:
            rel = path_obj.relative_to(raw_dir)
        except ValueError:
            rel = path_obj
        rel_str = str(rel).replace('\\', '/')
        if group_level == 'session':
            parent = str(rel.parent).replace('\\', '/')
            return parent if parent not in {'', '.'} else rel_str
        return rel_str

    mp_hands = get_mediapipe_hands_module()
    with mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=1,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
        model_complexity=1,
    ) as hands_detector:
        for class_id, class_dir in enumerate(class_dirs):
            class_to_id[class_dir.name] = class_id
            media_files = sorted([path for path in class_dir.rglob('*') if path.suffix.lower() in image_ext | video_ext])

            per_class_samples = []
            per_class_source_ids = []
            for media_path in media_files:
                suffix = media_path.suffix.lower()
                source_id = make_source_id(media_path)
                if suffix in image_ext:
                    lm = extract_landmarks_from_image(media_path, hands_detector)
                    if lm is not None:
                        per_class_samples.append(lm)
                        per_class_source_ids.append(source_id)
                elif suffix in video_ext:
                    lm_seq = extract_landmarks_from_video(media_path, hands_detector, frame_stride=frame_stride)
                    per_class_samples.extend(lm_seq)
                    per_class_source_ids.extend([source_id] * len(lm_seq))

            if len(per_class_samples) < min_samples_per_class:
                raise ValueError(
                    f'Class {class_dir.name} has only {len(per_class_samples)} extracted samples. '
                    f'Need at least {min_samples_per_class}.'
                )

            X_list.extend(per_class_samples)
            y_list.extend([class_id] * len(per_class_samples))
            source_ids_list.extend(per_class_source_ids)
            print(f'Class {class_dir.name}: {len(per_class_samples)} extracted landmark samples')

    X = np.asarray(X_list, dtype=np.float32)
    y = np.asarray(y_list, dtype=np.int64)
    source_ids = np.asarray(source_ids_list, dtype=object)

    rng = np.random.default_rng(SEED)
    order = rng.permutation(len(X))
    X = X[order]
    y = y[order]
    source_ids = source_ids[order]
    return X, y, class_to_id, source_ids


def resolve_existing_asset(filename: str) -> Path | None:
    """Find a precomputed asset in local, Colab, or Kaggle dataset mounts."""
    candidate_roots = [
        Path('.'),
        Path('data'),
        Path('exai_precomputed_assets'),
        Path('/content'),
        Path('/content/drive/MyDrive'),
        Path('/kaggle/working'),
        Path('/kaggle/input'),
        Path('/kaggle/input/dataset3'),
        Path('/kaggle/input/dataset3/exai_precomputed_assets'),
    ]

    for root in candidate_roots:
        candidate = root / filename
        if candidate.exists():
            return candidate

    for root in candidate_roots:
        if not root.exists() or not root.is_dir():
            continue
        matches = list(root.rglob(filename))
        if matches:
            return matches[0]

    return None


def resolve_raw_data_dir(raw_dir: Path) -> Path:
    """Resolve raw data folder across local/Colab/Kaggle roots."""
    raw_dir = Path(raw_dir)
    if raw_dir.is_absolute() and raw_dir.exists():
        return raw_dir

    candidate_dirs = [
        raw_dir,
        Path.cwd() / raw_dir,
        Path('data/raw'),
        Path('/kaggle/input/dataset1/exai_kaggle_upload/data/raw'),
        Path('/kaggle/input/exai_kaggle_upload/data/raw'),
        Path('/content/data/raw'),
    ]

    for candidate in candidate_dirs:
        if candidate.exists():
            return candidate

    searched = ', '.join(str(candidate) for candidate in candidate_dirs)
    raise FileNotFoundError(f'Raw data directory not found. Searched: {searched}')


def grouped_train_test_split(
    X: np.ndarray,
    y: np.ndarray,
    groups: np.ndarray,
    test_size: float = TEST_SIZE,
    seed: int = SEED,
    max_tries: int = 32,
):
    """Split by groups to reduce leakage from same-source samples across train/test."""
    classes = np.unique(y)
    best = None
    best_coverage = -1

    for offset in range(max_tries):
        splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed + offset)
        train_idx, test_idx = next(splitter.split(X, y, groups=groups))
        train_classes = np.unique(y[train_idx])
        test_classes = np.unique(y[test_idx])
        coverage = len(np.intersect1d(classes, train_classes)) + len(np.intersect1d(classes, test_classes))

        if coverage > best_coverage:
            best_coverage = coverage
            best = (train_idx, test_idx)

        if len(train_classes) == len(classes) and len(test_classes) == len(classes):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]
            return X_train, X_test, y_train, y_test, train_idx, test_idx

    train_idx, test_idx = best
    print('Warning: grouped split could not preserve all classes in both train and test; using best-effort grouped split.')
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    return X_train, X_test, y_train, y_test, train_idx, test_idx


def evaluate_multiclass(model: nn.Module, X_eval: np.ndarray, y_eval: np.ndarray):
    preds = predict_labels(model, X_eval)
    acc = accuracy_score(y_eval, preds)
    bal_acc = balanced_accuracy_score(y_eval, preds)
    mcc = matthews_corrcoef(y_eval, preds)
    cm = confusion_matrix(y_eval, preds)
    return {
        'accuracy': float(acc),
        'balanced_accuracy': float(bal_acc),
        'mcc': float(mcc),
        'confusion_matrix': cm,
        'classification_report': classification_report(y_eval, preds, digits=4),
    }


def run_repeated_split_evaluation(
    X_values: np.ndarray,
    y_values: np.ndarray,
    source_ids: np.ndarray | None = None,
    n_runs: int = 5,
    test_size: float = TEST_SIZE,
    base_seed: int = SEED,
    epochs: int = 24,
    batch_size: int = 128,
    lr: float = 6e-4,
) -> pd.DataFrame:
    """Train/evaluate across independent splits and report mean/std with 95% CI."""
    rows = []
    for run_idx in range(n_runs):
        run_seed = base_seed + run_idx
        if source_ids is not None and len(source_ids) == len(X_values):
            X_tr, X_te, y_tr, y_te, _, _ = grouped_train_test_split(
                X_values, y_values, np.asarray(source_ids), test_size=test_size, seed=run_seed
            )
            split_mode = 'grouped'
        else:
            X_tr, X_te, y_tr, y_te = train_test_split(
                X_values, y_values, test_size=test_size, random_state=run_seed, stratify=y_values
            )
            split_mode = 'stratified'

        model_run, _ = train_model(X_tr, y_tr, epochs=epochs, batch_size=batch_size, lr=lr, seed=run_seed)
        metrics = evaluate_multiclass(model_run, X_te, y_te)
        rows.append(
            {
                'run': run_idx,
                'seed': run_seed,
                'split_mode': split_mode,
                'accuracy': float(metrics['accuracy']),
                'balanced_accuracy': float(metrics['balanced_accuracy']),
                'mcc': float(metrics['mcc']),
            }
        )
        print(
            f"[Repeated Eval] run={run_idx + 1}/{n_runs} seed={run_seed} split={split_mode} "
            f"acc={metrics['accuracy']:.4f} bal_acc={metrics['balanced_accuracy']:.4f} mcc={metrics['mcc']:.4f}"
        )

    df = pd.DataFrame(rows)
    summary = []
    for metric_name in ['accuracy', 'balanced_accuracy', 'mcc']:
        values = df[metric_name].to_numpy(dtype=np.float64)
        summary.append(
            {
                'metric': metric_name,
                'mean': float(values.mean()),
                'std': float(values.std(ddof=1) if len(values) > 1 else 0.0),
                'ci95_low': float(np.percentile(values, 2.5)),
                'ci95_high': float(np.percentile(values, 97.5)),
            }
        )

    print('\nRepeated split summary (independent splits):')
    display(pd.DataFrame(summary))
    return df


def train_model(X: np.ndarray, y: np.ndarray, epochs: int = 24, batch_size: int = 128, lr: float = 6e-4, seed: int = 42):
    """Train with validation monitoring and LR schedule for stronger final accuracy."""
    torch.manual_seed(seed)
    np.random.seed(seed)

    n_classes = int(np.max(y)) + 1
    X_train_local, X_val_local, y_train_local, y_val_local = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )

    train_ds = torch.utils.data.TensorDataset(
        torch.tensor(X_train_local, dtype=torch.float32),
        torch.tensor(y_train_local, dtype=torch.long),
    )
    val_ds = torch.utils.data.TensorDataset(
        torch.tensor(X_val_local, dtype=torch.float32),
        torch.tensor(y_val_local, dtype=torch.long),
    )

    train_loader = torch.utils.data.DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = torch.utils.data.DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = GestureTransformerClassifier(num_classes=n_classes).to(device)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.02)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=2e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)

    best_val_bal_acc = -1.0
    best_state = deepcopy(model.state_dict())
    stale_epochs = 0
    early_stop_patience = 7

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        running_correct = 0
        running_total = 0

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            running_loss += loss.item() * xb.size(0)
            running_correct += (logits.argmax(dim=1) == yb).sum().item()
            running_total += xb.size(0)

        train_loss = running_loss / max(1, running_total)
        train_acc = running_correct / max(1, running_total)

        model.eval()
        val_preds = []
        val_true = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                logits = model(xb)
                val_preds.extend(logits.argmax(dim=1).detach().cpu().numpy().tolist())
                val_true.extend(yb.detach().cpu().numpy().tolist())

        val_acc = accuracy_score(val_true, val_preds)
        val_bal_acc = balanced_accuracy_score(val_true, val_preds)
        scheduler.step(val_bal_acc)

        if val_bal_acc > best_val_bal_acc:
            best_val_bal_acc = val_bal_acc
            best_state = deepcopy(model.state_dict())
            stale_epochs = 0
        else:
            stale_epochs += 1

        current_lr = optimizer.param_groups[0]['lr']
        print(
            f'Epoch {epoch:02d}/{epochs} | train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | '
            f'val_acc={val_acc:.4f} | val_bal_acc={val_bal_acc:.4f} | lr={current_lr:.2e}'
        )

        if stale_epochs >= early_stop_patience:
            print('Early stopping triggered.')
            break

    model.load_state_dict(best_state)
    model.eval()
    return model, best_val_bal_acc


def export_required_files(output_dir: Path, X: np.ndarray, y: np.ndarray, model: nn.Module, class_to_id: dict | None = None, source_ids: np.ndarray | None = None):
    output_dir.mkdir(parents=True, exist_ok=True)

    x_path = output_dir / 'X_landmarks.npy'
    y_path = output_dir / 'y_labels.npy'
    model_path = output_dir / 'gesture_model_best.pt'

    np.save(x_path, X.astype(np.float32))
    np.save(y_path, y.astype(np.int64))
    if source_ids is not None and len(source_ids) == len(X):
        np.save(output_dir / 'sample_source_ids.npy', np.asarray(source_ids, dtype=object), allow_pickle=True)

    model_for_export = deepcopy(model).cpu()
    scripted = torch.jit.script(model_for_export)
    scripted.save(str(model_path))

    if class_to_id is not None:
        with (output_dir / 'class_map.json').open('w', encoding='utf-8') as handle:
            json.dump(class_to_id, handle, indent=2)

    print('\nExport complete:')
    print(f'- {x_path}')
    print(f'- {y_path}')
    print(f'- {model_path}')
    if source_ids is not None and len(source_ids) == len(X):
        print(f'- {output_dir / "sample_source_ids.npy"}')
    if class_to_id is not None:
        print(f'- {output_dir / "class_map.json"}')


RAW_DATA_DIR = Path('data/raw')
OUTPUT_DIR = get_output_dir()
SOURCE_GROUP_LEVEL = 'session'  # 'file' or 'session'

BUILD_FROM_RAW_IF_NEEDED = True
FORCE_REBUILD_FROM_RAW = False
TRAIN_MODEL = True
TARGET_TOP1_ACCURACY = 0.95
RUN_REPEATED_SPLIT_EVAL = False
REPEATED_EVAL_RUNS = 5
REPEATED_EVAL_EPOCHS = 24

x_file = OUTPUT_DIR / 'X_landmarks.npy'
y_file = OUTPUT_DIR / 'y_labels.npy'
source_file = OUTPUT_DIR / 'sample_source_ids.npy'
fallback_x_file = resolve_existing_asset('X_landmarks.npy')
fallback_y_file = resolve_existing_asset('y_labels.npy')
fallback_source_file = resolve_existing_asset('sample_source_ids.npy')
class_map = None
source_ids_data = None

if (not FORCE_REBUILD_FROM_RAW) and x_file.exists() and y_file.exists():
    X_data = np.load(x_file).astype(np.float32)
    y_data = np.load(y_file).astype(np.int64)
    if source_file.exists():
        source_ids_data = np.load(source_file, allow_pickle=True)
    print(f'Using existing landmark arrays: X={X_data.shape}, y={y_data.shape}')
elif (not FORCE_REBUILD_FROM_RAW) and fallback_x_file is not None and fallback_y_file is not None:
    X_data = np.load(fallback_x_file).astype(np.float32)
    y_data = np.load(fallback_y_file).astype(np.int64)
    if fallback_source_file is not None:
        source_ids_data = np.load(fallback_source_file, allow_pickle=True)
    print(f'Using discovered landmark arrays: X={X_data.shape}, y={y_data.shape}')
else:
    if not BUILD_FROM_RAW_IF_NEEDED:
        raise FileNotFoundError('Landmark arrays are missing and BUILD_FROM_RAW_IF_NEEDED is False.')
    RAW_DATA_DIR = resolve_raw_data_dir(RAW_DATA_DIR)
    X_data, y_data, class_map, source_ids_data = build_landmark_dataset_from_raw(
        raw_dir=RAW_DATA_DIR,
        min_samples_per_class=25,
        frame_stride=4,
        group_level=SOURCE_GROUP_LEVEL,
    )
    print(f'Built real landmark dataset: X={X_data.shape}, y={y_data.shape}')

if TRAIN_MODEL:
    trained_model, best_val_bal_acc = train_model(X_data, y_data, epochs=24, batch_size=128, lr=6e-4, seed=SEED)
    print(f'Best validation balanced accuracy: {best_val_bal_acc:.4f}')
    export_required_files(OUTPUT_DIR, X_data, y_data, trained_model, class_to_id=class_map, source_ids=source_ids_data)

# Keep variables available for following notebook sections.
X, y = X_data, y_data
if source_ids_data is not None and len(source_ids_data) == len(X):
    X_train, X_test, y_train, y_test, train_idx, test_idx = grouped_train_test_split(
        X, y, np.asarray(source_ids_data), test_size=TEST_SIZE, seed=SEED
    )
    print(f'Prepared grouped splits (source-aware): X_train={X_train.shape}, X_test={X_test.shape}')
else:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)
    print('Warning: sample_source_ids.npy not found. Falling back to sample-level split (higher leakage risk).')
    print(f'Prepared stratified sample-level splits: X_train={X_train.shape}, X_test={X_test.shape}')

if TRAIN_MODEL:
    strict_eval = evaluate_multiclass(trained_model, X_test, y_test)
    print('\nStrict multiclass holdout metrics')
    print(f"Top-1 Accuracy: {strict_eval['accuracy']:.4f}")
    print(f"Balanced Accuracy: {strict_eval['balanced_accuracy']:.4f}")
    print(f"MCC: {strict_eval['mcc']:.4f}")
    print('Classification report:')
    print(strict_eval['classification_report'])
    print('Confusion matrix:')
    print(strict_eval['confusion_matrix'])
    if strict_eval['accuracy'] < TARGET_TOP1_ACCURACY:
        print(
            f"WARNING: accuracy {strict_eval['accuracy']:.4f} is below target {TARGET_TOP1_ACCURACY:.2f}. "
            'Increase data quality/quantity or tune model hyperparameters for stricter performance goals.'
        )

if TRAIN_MODEL and RUN_REPEATED_SPLIT_EVAL:
    repeated_eval_df = run_repeated_split_evaluation(
        X_values=X,
        y_values=y,
        source_ids=source_ids_data,
        n_runs=REPEATED_EVAL_RUNS,
        test_size=TEST_SIZE,
        base_seed=SEED,
        epochs=REPEATED_EVAL_EPOCHS,
        batch_size=128,
        lr=6e-4,
    )

AttributeError: module 'mediapipe' has no attribute 'solutions'

## SHAP Stability Analysis

This section computes KernelExplainer SHAP values on a fixed class-balanced background and repeats the analysis under multiple random seeds. The goal is to see whether the same joints stay important across runs.

In [ ]:
# SHAP utilities and stability analysis

def select_class_balanced_indices(y_values: np.ndarray, n_total: int, seed: int = 0) -> np.ndarray:
    """Select roughly class-balanced indices with replacement only if needed."""
    rng = np.random.default_rng(seed)
    classes = np.unique(y_values)
    per_class = n_total // len(classes)
    remainder = n_total % len(classes)
    selected = []

    for class_index, class_label in enumerate(classes):
        class_indices = np.where(y_values == class_label)[0]
        n_take = per_class + (1 if class_index < remainder else 0)
        if len(class_indices) == 0:
            continue
        replace = len(class_indices) < n_take
        chosen = rng.choice(class_indices, size=n_take, replace=replace)
        selected.extend(chosen.tolist())

    selected = np.array(selected, dtype=int)
    if len(selected) < n_total:
        remaining = np.setdiff1d(np.arange(len(y_values)), selected, assume_unique=False)
        extra = rng.choice(remaining, size=n_total - len(selected), replace=False)
        selected = np.concatenate([selected, extra])

    rng.shuffle(selected)
    return selected[:n_total]


def select_eval_subset(X_values: np.ndarray, y_values: np.ndarray, n_samples: int, seed: int = 0):
    """Sample a deterministic evaluation subset from the test split."""
    rng = np.random.default_rng(seed)
    n_samples = min(n_samples, len(X_values))
    indices = rng.choice(len(X_values), size=n_samples, replace=False)
    return X_values[indices], y_values[indices], indices


def make_shap_cache_key(model: nn.Module, background_flat: np.ndarray, eval_flat: np.ndarray, nsamples: int) -> str:
    """Create a stable cache key for a SHAP run."""
    hasher = hashlib.sha1()
    hasher.update(type(model).__name__.encode('utf-8'))
    hasher.update(np.ascontiguousarray(background_flat).view(np.uint8))
    hasher.update(np.ascontiguousarray(eval_flat).view(np.uint8))
    hasher.update(str(nsamples).encode('utf-8'))
    return hasher.hexdigest()


def compute_shap(model: nn.Module, background: np.ndarray, eval_samples: np.ndarray, nsamples: int = SHAP_NSAMPLES):
    """Compute KernelExplainer SHAP values on flattened landmark inputs.

    Returns a list of SHAP arrays, one per class for multi-class models.
    """
    background_flat = background.reshape(len(background), -1)
    eval_flat = eval_samples.reshape(len(eval_samples), -1)

    cache_key = make_shap_cache_key(model, background_flat, eval_flat, nsamples)
    cache_path = SHAP_CACHE_DIR / f'{cache_key}.pkl'
    if SHAP_CACHE_ENABLED and cache_path.exists():
        with cache_path.open('rb') as handle:
            cached = pickle.load(handle)
        return None, cached['shap_values']

    explainer = shap.KernelExplainer(
        lambda x: predict_proba_flat(model, x, batch_size=256, device=DEVICE),
        background_flat,
    )
    shap_values = explainer.shap_values(eval_flat, nsamples=nsamples)

    if SHAP_CACHE_ENABLED:
        SHAP_CACHE_DIR.mkdir(parents=True, exist_ok=True)
        with cache_path.open('wb') as handle:
            pickle.dump({'shap_values': shap_values}, handle)

    return explainer, shap_values


def _extract_shap_sample_class(shap_values, sample_index: int, class_index: int) -> np.ndarray:
    """Extract class-specific SHAP vector robustly across SHAP output formats."""
    if isinstance(shap_values, list):
        return np.asarray(shap_values[class_index][sample_index], dtype=np.float32).reshape(-1)

    arr = np.asarray(shap_values)
    if arr.ndim == 3:
        # Typical new SHAP format: (n_samples, n_features, n_classes)
        if arr.shape[1] == INPUT_DIM:
            return np.asarray(arr[sample_index, :, class_index], dtype=np.float32).reshape(-1)
        # Alternative layout: (n_samples, n_classes, n_features)
        if arr.shape[2] == INPUT_DIM:
            return np.asarray(arr[sample_index, class_index, :], dtype=np.float32).reshape(-1)

    if arr.ndim == 2:
        return np.asarray(arr[sample_index], dtype=np.float32).reshape(-1)

    return np.asarray(arr[sample_index], dtype=np.float32).reshape(-1)


def get_sample_shap_vector(shap_values, sample_index: int, class_index: int) -> np.ndarray:
    """Extract a single flattened SHAP vector for the requested sample and class."""
    vector = _extract_shap_sample_class(shap_values, sample_index, class_index)
    if len(vector) == INPUT_DIM:
        return vector
    if len(vector) == NUM_JOINTS and NUM_COORDS == 3:
        return np.repeat(vector / float(NUM_COORDS), NUM_COORDS)
    if len(vector) > INPUT_DIM:
        return vector[:INPUT_DIM]
    pad = np.zeros(INPUT_DIM, dtype=np.float32)
    pad[:len(vector)] = vector
    return pad


def aggregate_joint_importance(flat_shap_values: np.ndarray) -> np.ndarray:
    """Aggregate flattened SHAP values into joint-level importance scores.

    Input can be shape (63,) or (N, 63). The return shape is (21,) or (N, 21).
    """
    flat_shap_values = np.asarray(flat_shap_values, dtype=np.float32)
    if flat_shap_values.ndim == 1:
        flat_shap_values = flat_shap_values[None, :]
    joint_scores = np.abs(flat_shap_values).reshape(-1, NUM_JOINTS, NUM_COORDS).sum(axis=2)
    return joint_scores.squeeze()


def ensure_joint_vector(values: np.ndarray) -> np.ndarray:
    """Force an attribution-like vector into joint space (length 21)."""
    arr = np.asarray(values, dtype=np.float32)

    if arr.ndim == 1:
        if len(arr) == NUM_JOINTS:
            return arr
        if len(arr) == INPUT_DIM:
            return aggregate_joint_importance(arr)
        if len(arr) > NUM_JOINTS:
            trimmed = arr[: (len(arr) // NUM_JOINTS) * NUM_JOINTS]
            if len(trimmed) >= NUM_JOINTS:
                reshaped = trimmed.reshape(-1, NUM_JOINTS)
                return reshaped.mean(axis=0)
        out = np.zeros(NUM_JOINTS, dtype=np.float32)
        out[:min(NUM_JOINTS, len(arr))] = arr[:min(NUM_JOINTS, len(arr))]
        return out

    if arr.ndim >= 2:
        if arr.shape[-1] == NUM_JOINTS:
            return arr.reshape(-1, NUM_JOINTS).mean(axis=0)
        if arr.shape[-1] == INPUT_DIM:
            aggregated = aggregate_joint_importance(arr.reshape(-1, INPUT_DIM))
            return np.asarray(aggregated, dtype=np.float32).reshape(-1, NUM_JOINTS).mean(axis=0)

    return np.zeros(NUM_JOINTS, dtype=np.float32)


def top_k_overlap(run_joint_importances: np.ndarray, k: int = 5) -> float:
    """Average overlap of top-k joints across runs."""
    top_sets = []
    for row in run_joint_importances:
        row_array = ensure_joint_vector(row)
        top_sets.append(set(np.argsort(-row_array)[:k].tolist()))
    overlaps = []
    for i in range(len(top_sets)):
        for j in range(i + 1, len(top_sets)):
            overlaps.append(len(top_sets[i] & top_sets[j]) / float(k))
    return float(np.mean(overlaps)) if overlaps else 0.0


def _top2_class_indices(prob_row: np.ndarray) -> tuple[int, int]:
    order = np.argsort(np.asarray(prob_row, dtype=np.float32))[::-1]
    if len(order) == 1:
        return int(order[0]), int(order[0])
    return int(order[0]), int(order[1])


def plot_joint_importance_with_errorbars(
    mean_importance: np.ndarray,
    std_importance: np.ndarray,
    title: str = 'Mean SHAP importance per joint',
    highlight_top_n: int = 3,
):
    mean_importance = ensure_joint_vector(mean_importance)
    std_importance = ensure_joint_vector(std_importance)
    n_items = min(len(mean_importance), len(std_importance), NUM_JOINTS)
    mean_importance = mean_importance[:n_items]
    std_importance = std_importance[:n_items]
    joint_labels = get_joint_labels(n_items)
    order = np.argsort(-mean_importance)

    colors = ['#4C78A8'] * n_items
    for rank, idx in enumerate(order[:min(highlight_top_n, n_items)]):
        colors[idx] = '#E45756' if rank == 0 else '#F58518' if rank == 1 else '#72B7B2'

    plt.figure(figsize=(14, 6))
    bars = plt.bar(np.array(joint_labels)[order], mean_importance[order], yerr=std_importance[order], capsize=4, color=np.array(colors)[order])
    plt.ylabel('Aggregated SHAP importance')
    plt.xlabel('Joint')
    plt.title(title)
    plt.xticks(rotation=45, ha='right')

    for bar in bars[:min(highlight_top_n, len(bars))]:
        height = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width() / 2.0,
            height,
            f'{height:.3f}',
            ha='center',
            va='bottom',
            fontsize=9,
            rotation=0,
        )

    plt.tight_layout()
    plt.show()


def run_shap_stability_analysis(
    model: nn.Module,
    X_train_values: np.ndarray,
    y_train_values: np.ndarray,
    X_test_values: np.ndarray,
    y_test_values: np.ndarray,
    background_size: int = N_BACKGROUND,
    eval_size: int = N_SHAP_EVAL,
    seeds = SHAP_RUN_SEEDS,
    nsamples: int = SHAP_NSAMPLES,
):
    """Repeat SHAP analysis under multiple random seeds and summarize joint stability."""
    all_run_joint_importances = []
    all_run_runnerup_importances = []
    all_run_contrast_importances = []
    all_run_details = []

    for run_seed in seeds:
        set_seed(run_seed)
        background_idx = select_class_balanced_indices(y_train_values, background_size, seed=run_seed)
        eval_X, eval_y, eval_idx = select_eval_subset(X_test_values, y_test_values, eval_size, seed=run_seed)
        background = X_train_values[background_idx]

        _, shap_values = compute_shap(model, background, eval_X, nsamples=nsamples)
        probs = predict_proba(model, eval_X)

        sample_joint_importances = []
        sample_runnerup_importances = []
        sample_contrast_importances = []
        sample_confidence_gaps = []

        for i in range(len(eval_X)):
            pred_idx, runnerup_idx = _top2_class_indices(probs[i])
            sample_pred_shap = get_sample_shap_vector(shap_values, i, pred_idx)
            sample_runnerup_shap = get_sample_shap_vector(shap_values, i, runnerup_idx)

            sample_pred_joint = ensure_joint_vector(aggregate_joint_importance(sample_pred_shap))
            sample_runnerup_joint = ensure_joint_vector(aggregate_joint_importance(sample_runnerup_shap))
            sample_contrast_joint = np.abs(sample_pred_joint - sample_runnerup_joint)

            sample_joint_importances.append(sample_pred_joint)
            sample_runnerup_importances.append(sample_runnerup_joint)
            sample_contrast_importances.append(sample_contrast_joint)
            sample_confidence_gaps.append(float(probs[i, pred_idx] - probs[i, runnerup_idx]))

        sample_joint_importances = np.asarray(sample_joint_importances, dtype=np.float32)
        sample_runnerup_importances = np.asarray(sample_runnerup_importances, dtype=np.float32)
        sample_contrast_importances = np.asarray(sample_contrast_importances, dtype=np.float32)

        mean_joint_importance = ensure_joint_vector(sample_joint_importances.mean(axis=0))
        mean_runnerup_importance = ensure_joint_vector(sample_runnerup_importances.mean(axis=0))
        mean_contrast_importance = ensure_joint_vector(sample_contrast_importances.mean(axis=0))

        all_run_joint_importances.append(mean_joint_importance)
        all_run_runnerup_importances.append(mean_runnerup_importance)
        all_run_contrast_importances.append(mean_contrast_importance)
        all_run_details.append(
            {
                'seed': run_seed,
                'background_idx': background_idx,
                'eval_idx': eval_idx,
                'eval_y': eval_y,
                'sample_joint_importances': sample_joint_importances,
                'sample_runnerup_importances': sample_runnerup_importances,
                'sample_contrast_importances': sample_contrast_importances,
                'sample_confidence_gap': np.asarray(sample_confidence_gaps, dtype=np.float32),
                'mean_joint_importance': mean_joint_importance,
                'mean_runnerup_importance': mean_runnerup_importance,
                'mean_contrast_importance': mean_contrast_importance,
            }
        )

        print(f'[SHAP] Seed {run_seed}: computed SHAP for {len(eval_X)} samples with background size {len(background)}')

    all_run_joint_importances = np.asarray(all_run_joint_importances, dtype=np.float32)
    all_run_runnerup_importances = np.asarray(all_run_runnerup_importances, dtype=np.float32)
    all_run_contrast_importances = np.asarray(all_run_contrast_importances, dtype=np.float32)
    mean_importance = ensure_joint_vector(all_run_joint_importances.mean(axis=0))
    std_importance = ensure_joint_vector(all_run_joint_importances.std(axis=0))
    mean_runnerup_importance = ensure_joint_vector(all_run_runnerup_importances.mean(axis=0))
    mean_contrast_importance = ensure_joint_vector(all_run_contrast_importances.mean(axis=0))

    print('\nSHAP stability summary')
    print('Top-5 joint overlap across runs:', round(top_k_overlap(all_run_joint_importances, k=5), 4))
    print('Top-5 joints by mean importance:', np.argsort(-mean_importance)[:5].tolist())
    print('Top-5 joints by pred-vs-runnerup contrast:', np.argsort(-mean_contrast_importance)[:5].tolist())

    return {
        'run_joint_importances': all_run_joint_importances,
        'run_runnerup_importances': all_run_runnerup_importances,
        'run_contrast_importances': all_run_contrast_importances,
        'mean_importance': mean_importance,
        'std_importance': std_importance,
        'mean_runnerup_importance': mean_runnerup_importance,
        'mean_contrast_importance': mean_contrast_importance,
        'run_details': all_run_details,
    }


# Example call for the SHAP stability experiment.
# Run this cell when you want the full analysis; it is intentionally not executed automatically.
RUN_SHAP_STABILITY = True
if RUN_SHAP_STABILITY:
    shap_stability_results = run_shap_stability_analysis(
        model=load_model(),
        X_train_values=X_train,
        y_train_values=y_train,
        X_test_values=X_test,
        y_test_values=y_test,
    )
    plot_joint_importance_with_errorbars(
        shap_stability_results['mean_importance'],
        shap_stability_results['std_importance'],
        title='SHAP stability across seeds: mean ± std per joint',
        highlight_top_n=3,
    )

### Interpretation note

If the same few joints remain on top across seeds and the error bars stay small relative to the mean, the SHAP attribution pattern is stable. Large standard deviations or frequent ranking changes suggest the explanation is fragile to background selection or sampling noise.

## Fragility, Robustness, and Negative Motion

This section first measures how often predictions flip under Gaussian noise, then generates non-intent inputs to compare confidence, SHAP concentration, and Gini score distributions against real gestures.

In [ ]:
# Robustness analysis, real negative-motion support, and Gini computation

FINGER_GROUPS = {
    'thumb': [1, 2, 3, 4],
    'index': [5, 6, 7, 8],
    'middle': [9, 10, 11, 12],
    'ring': [13, 14, 15, 16],
    'pinky': [17, 18, 19, 20],
}


def add_noise(X_values: np.ndarray, sigma: float, joint_index: int | None = None, seed: int = 0) -> np.ndarray:
    """Add Gaussian noise to all joints or to one selected joint only."""
    if sigma <= 0:
        return X_values.copy().astype(np.float32)

    rng = np.random.default_rng(seed)
    noisy = X_values.copy().astype(np.float32)

    if joint_index is None:
        noisy += rng.normal(0.0, sigma, size=noisy.shape).astype(np.float32)
    else:
        joint_noise = rng.normal(0.0, sigma, size=(len(noisy), NUM_COORDS)).astype(np.float32)
        noisy[:, joint_index, :] += joint_noise

    return noisy


def apply_small_rigid_transform(X_values: np.ndarray, seed: int = 0, max_angle_deg: float = 10.0, max_translation: float = 0.03) -> np.ndarray:
    """Apply a small rotation and translation around the wrist to mimic natural motion."""
    rng = np.random.default_rng(seed)
    transformed = X_values.copy().astype(np.float32)
    angles = np.deg2rad(rng.uniform(-max_angle_deg, max_angle_deg, size=3))
    cx, cy, cz = np.cos(angles)
    sx, sy, sz = np.sin(angles)

    rx = np.array([[1, 0, 0], [0, cx, -sx], [0, sx, cx]], dtype=np.float32)
    ry = np.array([[cy, 0, sy], [0, 1, 0], [-sy, 0, cy]], dtype=np.float32)
    rz = np.array([[cz, -sz, 0], [sz, cz, 0], [0, 0, 1]], dtype=np.float32)
    rotation = rz @ ry @ rx

    translation = rng.uniform(-max_translation, max_translation, size=(1, 1, 3)).astype(np.float32)
    wrist = transformed[:, :1, :]
    centered = transformed - wrist
    transformed = centered @ rotation.T + wrist + translation
    return transformed.astype(np.float32)


def generate_partial_gesture_noise(X_values: np.ndarray, seed: int = 0) -> np.ndarray:
    """Simulate incomplete gestures by retracting fingers toward the wrist."""
    rng = np.random.default_rng(seed)
    partial = X_values.copy().astype(np.float32)
    wrist = partial[:, :1, :]

    for joint_indices in FINGER_GROUPS.values():
        progress = rng.uniform(0.15, 0.75, size=(len(partial), 1, 1)).astype(np.float32)
        finger = partial[:, joint_indices, :]
        finger = wrist + (finger - wrist) * progress
        finger += rng.normal(0.0, 0.015, size=finger.shape).astype(np.float32)
        partial[:, joint_indices, :] = finger

    return partial


def generate_finger_motion_noise(X_values: np.ndarray, seed: int = 0) -> np.ndarray:
    """Simulate finger-specific motion with correlated offsets across each finger chain."""
    rng = np.random.default_rng(seed)
    finger_motion = X_values.copy().astype(np.float32)

    for joint_indices in FINGER_GROUPS.values():
        base_direction = rng.normal(0.0, 0.04, size=(len(finger_motion), 1, NUM_COORDS)).astype(np.float32)
        weights = np.linspace(0.3, 1.0, num=len(joint_indices), dtype=np.float32).reshape(1, -1, 1)
        correlated_offset = base_direction * weights
        finger_motion[:, joint_indices, :] += correlated_offset
        finger_motion[:, joint_indices, :] += rng.normal(0.0, 0.01, size=finger_motion[:, joint_indices, :].shape).astype(np.float32)

    return finger_motion


def compute_gini(values: np.ndarray, eps: float = 1e-8) -> float:
    """Compute the Gini coefficient over a normalized importance distribution."""
    x = normalize_importance_vector(values, eps=eps)
    if np.allclose(x.sum(), 0.0):
        return 0.0
    x = np.sort(x)
    n = len(x)
    index = np.arange(1, n + 1, dtype=np.float64)
    gini = (2.0 * np.sum(index * x) / max(n * (x.sum() + eps), eps)) - (n + 1.0) / n
    return float(np.clip(gini, 0.0, 1.0))


def run_robustness_analysis(
    model: nn.Module,
    X_eval: np.ndarray,
    sigmas = ROBUSTNESS_SIGMAS,
    n_trials: int = ROBUSTNESS_N_TRIALS,
    max_samples: int = ROBUSTNESS_MAX_SAMPLES,
    seed: int = SEED,
):
    """Measure how often predictions flip under noisy perturbations."""
    rng = np.random.default_rng(seed)
    n_eval = min(max_samples, len(X_eval))
    sample_indices = rng.choice(len(X_eval), size=n_eval, replace=False)
    X_subset = X_eval[sample_indices]
    base_preds = predict_labels(model, X_subset)

    sample_flip_prob = np.zeros((n_eval, len(sigmas)), dtype=np.float32)
    joint_flip_prob = np.zeros((len(sigmas), NUM_JOINTS), dtype=np.float32)

    for sigma_index, sigma in enumerate(sigmas):
        for sample_index in range(n_eval):
            flips = 0
            x_clean = X_subset[sample_index:sample_index + 1]
            for trial in range(n_trials):
                noisy = add_noise(x_clean, sigma=sigma, seed=seed + 1000 * sigma_index + 10 * sample_index + trial)
                pred = predict_labels(model, noisy)[0]
                flips += int(pred != base_preds[sample_index])
            sample_flip_prob[sample_index, sigma_index] = flips / float(n_trials)

        for joint_index in range(NUM_JOINTS):
            flips = 0
            total = 0
            for sample_index in range(n_eval):
                x_clean = X_subset[sample_index:sample_index + 1]
                for trial in range(n_trials):
                    noisy = add_noise(
                        x_clean,
                        sigma=sigma,
                        joint_index=joint_index,
                        seed=seed + 5000 * sigma_index + 100 * joint_index + 10 * sample_index + trial,
                    )
                    pred = predict_labels(model, noisy)[0]
                    flips += int(pred != base_preds[sample_index])
                    total += 1
            joint_flip_prob[sigma_index, joint_index] = flips / float(total)

        print(f'[Robustness] sigma={sigma:.3f} completed')

    overall_curve = sample_flip_prob.mean(axis=0)
    overall_std = sample_flip_prob.std(axis=0)
    mean_joint_fragility = joint_flip_prob.mean(axis=0)
    top_fragile_joints = np.argsort(-mean_joint_fragility)[:5]

    return {
        'sample_indices': sample_indices,
        'base_preds': base_preds,
        'sample_flip_prob': sample_flip_prob,
        'joint_flip_prob': joint_flip_prob,
        'overall_curve': overall_curve,
        'overall_std': overall_std,
        'mean_joint_fragility': mean_joint_fragility,
        'top_fragile_joints': top_fragile_joints,
        'sigmas': np.asarray(sigmas, dtype=np.float32),
    }


def plot_robustness_curve(sigmas: np.ndarray, overall_curve: np.ndarray, overall_std: np.ndarray):
    plt.figure(figsize=(10, 5))
    plt.errorbar(sigmas, overall_curve, yerr=overall_std, marker='o', linewidth=2, capsize=4, color='#F58518')
    plt.xlabel('Noise sigma')
    plt.ylabel('Prediction flip probability')
    plt.title('Gaussian noise robustness: flip probability vs sigma')
    plt.ylim(0.0, 1.0)
    plt.tight_layout()
    plt.show()


def plot_sample_flip_heatmap(sample_flip_prob: np.ndarray, sigmas: np.ndarray):
    plt.figure(figsize=(12, 6))
    sns.heatmap(sample_flip_prob, cmap='mako', vmin=0.0, vmax=1.0, xticklabels=[f'{s:.2f}' for s in sigmas])
    plt.xlabel('Sigma')
    plt.ylabel('Sample index')
    plt.title('Per-sample flip probability under Gaussian noise')
    plt.tight_layout()
    plt.show()


def plot_joint_fragility_heatmap(joint_flip_prob: np.ndarray, sigmas: np.ndarray):
    plt.figure(figsize=(14, 6))
    joint_labels = get_joint_labels(NUM_JOINTS)
    sns.heatmap(
        joint_flip_prob,
        cmap='rocket_r',
        vmin=0.0,
        vmax=1.0,
        xticklabels=joint_labels,
        yticklabels=[f'{s:.2f}' for s in sigmas],
    )
    plt.xlabel('Joint')
    plt.ylabel('Sigma')
    plt.title('Per-joint fragility heatmap: flip probability when one joint is perturbed')
    plt.tight_layout()
    plt.show()


def plot_fragility_summary(mean_joint_fragility: np.ndarray, top_n: int = 5):
    labels = get_joint_labels(len(mean_joint_fragility))
    order = np.argsort(-mean_joint_fragility)
    plt.figure(figsize=(14, 6))
    colors = ['#72B7B2'] * len(mean_joint_fragility)
    for rank, idx in enumerate(order[:min(top_n, len(order))]):
        colors[idx] = '#E45756' if rank == 0 else '#F58518' if rank == 1 else '#4C78A8'
    bars = plt.bar(np.array(labels)[order], mean_joint_fragility[order], color=np.array(colors)[order])
    plt.xticks(rotation=45, ha='right')
    plt.ylabel('Mean flip probability')
    plt.xlabel('Joint')
    plt.title('Most fragile joints under perturbation')
    for bar in bars[:min(top_n, len(bars))]:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width() / 2.0, height, f'{height:.3f}', ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    plt.show()


def resolve_random_motion_dir(random_motion_dir: Path) -> Path:
    """Resolve the random-motion directory from common workspace locations."""
    candidate_dirs = []
    path_value = Path(random_motion_dir)
    if path_value.is_absolute():
        candidate_dirs.append(path_value)
    else:
        candidate_dirs.extend([path_value, Path.cwd() / path_value])

    relative_candidates = [
        Path('data/random_motion'),
        Path('exai_kaggle_upload/data/random_motion'),
    ]
    search_roots = [
        Path('/kaggle/working'),
        Path('/kaggle/input'),
        Path('/kaggle/input/dataset1'),
        Path('/kaggle/input/dataset1/exai_kaggle_upload'),
        Path('/kaggle/input/dataset1/exai_kaggle_upload/data'),
        Path('/kaggle/input/dataset1/exai_kaggle_upload/data/random_motion'),
        Path('/kaggle/input/dataset3'),
        Path('/kaggle/input/dataset3/exai_precomputed_assets'),
        Path('/kaggle/input/exai_kaggle_upload'),
        Path('/kaggle/input/exai_kaggle_upload/data'),
        Path('/kaggle/input/exai_kaggle_upload/data/random_motion'),
        Path('/mnt/c/Users/Ronit/Desktop/Exai'),
        Path('/c/Users/Ronit/Desktop/Exai'),
    ]

    for root in search_roots:
        for relative_candidate in relative_candidates:
            candidate_dirs.append(root / relative_candidate)

    # Last resort for hosted datasets: find any folder named random_motion.
    for root in search_roots:
        if not root.exists() or not root.is_dir():
            continue
        try:
            matches = [path for path in root.rglob('random_motion') if path.is_dir()]
            candidate_dirs.extend(matches)
        except OSError:
            continue

    seen = set()
    unique_candidates = []
    for candidate in candidate_dirs:
        key = str(candidate)
        if key not in seen:
            seen.add(key)
            unique_candidates.append(candidate)

    for candidate in unique_candidates:
        if candidate.exists():
            return candidate
    searched = ', '.join(str(candidate) for candidate in unique_candidates)
    raise FileNotFoundError(f'Random motion directory not found. Searched: {searched}')


def extract_random_motion_landmarks(random_motion_dir: Path, n_samples: int, frame_stride: int = 4, seed: int = SEED) -> np.ndarray | None:
    """Extract real non-intent hand-motion landmarks from data/random_motion."""
    random_motion_dir = resolve_random_motion_dir(random_motion_dir)

    image_ext = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    video_ext = {'.mp4', '.avi', '.mov', '.mkv'}
    media_files = sorted([path for path in random_motion_dir.rglob('*') if path.suffix.lower() in image_ext | video_ext])
    if not media_files:
        raise ValueError(f'No media files found under {random_motion_dir}')

    mp_hands = get_mediapipe_hands_module()
    if mp_hands is None or not hasattr(mp_hands, 'Hands'):
        print('MediaPipe Hands is unavailable in this kernel; using landmark-space fallback negatives.')
        return None

    extracted = []
    with mp_hands.Hands(
        static_image_mode=False,
        max_num_hands=1,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5,
        model_complexity=1,
    ) as hands_detector:
        for media_path in media_files:
            if media_path.suffix.lower() in image_ext:
                lm = extract_landmarks_from_image(media_path, hands_detector)
                if lm is not None:
                    extracted.append(lm)
            else:
                seq = extract_landmarks_from_video(media_path, hands_detector, frame_stride=frame_stride)
                extracted.extend(seq)
            if len(extracted) >= n_samples * 2:
                break

    if len(extracted) < n_samples:
        raise ValueError(
            f'Only {len(extracted)} real random-motion landmarks extracted, need >= {n_samples}. '
            'Add more files in data/random_motion.'
        )

    rng = np.random.default_rng(seed)
    idx = rng.choice(len(extracted), size=n_samples, replace=False)
    return np.asarray(extracted, dtype=np.float32)[idx]


def generate_negative_samples(
    X_reference: np.ndarray,
    n_samples: int = NEGATIVE_SAMPLES_PER_TYPE,
    seed: int = SEED,
    strict_real_negative: bool = True,
    random_motion_dir: Path = Path('data/random_motion'),
    allow_fallback_if_missing: bool = True,
):
    """Create negative samples. In strict mode, requires real random-motion captures."""
    rng = np.random.default_rng(seed)
    n_samples = min(n_samples, len(X_reference))
    base_indices = rng.choice(len(X_reference), size=n_samples, replace=False)
    base = X_reference[base_indices].astype(np.float32)

    negatives = {'real': base}

    if strict_real_negative:
        try:
            real_random = extract_random_motion_landmarks(random_motion_dir, n_samples=n_samples, seed=seed + 10)
            if real_random is None:
                raise RuntimeError('MediaPipe Hands unavailable.')
            negatives['real_random_motion'] = real_random
        except Exception as err:
            if allow_fallback_if_missing:
                print(f'Warning: {err}. Falling back to landmark-space negative motion.')
                negatives['fallback_random_motion'] = generate_finger_motion_noise(generate_partial_gesture_noise(base, seed=seed + 10), seed=seed + 11)
            else:
                raise RuntimeError(
                    'Strict real-negative evaluation failed. Provide usable MediaPipe Hands and real data/random_motion files.'
                ) from err
    else:
        negatives['partial_gesture'] = generate_partial_gesture_noise(base, seed=seed + 1)
        negatives['finger_motion'] = generate_finger_motion_noise(base, seed=seed + 2)
        negatives['rigid_motion'] = apply_small_rigid_transform(base, seed=seed + 3)

    return negatives


def _shap_and_gini_for_batch(model: nn.Module, X_batch: np.ndarray, background: np.ndarray, nsamples: int, seed: int = 0):
    """Compute SHAP, joint importance, Gini, confidence, and entropy for a batch."""
    set_seed(seed)
    _, shap_values = compute_shap(model, background, X_batch, nsamples=nsamples)
    preds = predict_labels(model, X_batch)
    probs = predict_proba(model, X_batch)
    conf = probs.max(axis=1)
    entropy = -np.sum(probs * np.log(probs + 1e-12), axis=1)

    joint_importances = []
    normalized_joint_importances = []
    contrast_importances = []
    ginis = []
    for i in range(len(X_batch)):
        pred_class = int(preds[i])
        pred_shap = get_sample_shap_vector(shap_values, i, pred_class)
        pred_joint = normalize_importance_vector(aggregate_joint_importance(pred_shap))

        class_order = np.argsort(np.asarray(probs[i], dtype=np.float32))[::-1]
        runnerup_idx = int(class_order[1]) if len(class_order) > 1 else pred_class
        runnerup_shap = get_sample_shap_vector(shap_values, i, runnerup_idx)
        runnerup_joint = normalize_importance_vector(aggregate_joint_importance(runnerup_shap))

        contrast = float(np.sum(np.abs(pred_joint - runnerup_joint)) / 2.0)
        joint_importances.append(pred_joint)
        normalized_joint_importances.append(pred_joint)
        contrast_importances.append(contrast)
        ginis.append(compute_gini(pred_joint))

    return {
        'preds': preds,
        'probs': probs,
        'confidence': conf,
        'entropy': entropy,
        'joint_importance': np.asarray(joint_importances, dtype=np.float32),
        'normalized_joint_importance': np.asarray(normalized_joint_importances, dtype=np.float32),
        'contrast': np.asarray(contrast_importances, dtype=np.float32),
        'gini': np.asarray(ginis, dtype=np.float32),
    }


def run_negative_motion_analysis(
    model: nn.Module,
    X_train_values: np.ndarray,
    y_train_values: np.ndarray,
    X_real_eval: np.ndarray,
    n_negative: int = NEGATIVE_SAMPLES_PER_TYPE,
    n_shap: int = NEGATIVE_SHAP_SAMPLES,
    seed: int = SEED,
    strict_real_negative: bool = True,
    random_motion_dir: Path = Path('data/random_motion'),
    require_real_random_motion: bool = False,
):
    """Compare real gestures and negative motion under confidence, SHAP, contrast, and Gini."""
    rng = np.random.default_rng(seed)
    n_negative = min(n_negative, len(X_real_eval))
    eval_idx = rng.choice(len(X_real_eval), size=n_negative, replace=False)
    real_subset = X_real_eval[eval_idx].astype(np.float32)

    background_idx = select_class_balanced_indices(y_train_values, N_BACKGROUND, seed=seed)
    background = X_train_values[background_idx]

    negatives = generate_negative_samples(
        X_reference=real_subset,
        n_samples=n_negative,
        seed=seed,
        strict_real_negative=strict_real_negative,
        random_motion_dir=random_motion_dir,
        allow_fallback_if_missing=(not require_real_random_motion),
    )

    if require_real_random_motion and 'real_random_motion' not in negatives:
        raise RuntimeError(
            'Strict Challenge-2 evaluation requires true real_random_motion negatives. Fallback negatives are not allowed.'
        )

    results = {}
    for label, samples in negatives.items():
        sample_count = min(n_shap, len(samples))
        samples_for_shap = samples[:sample_count]
        batch_stats = _shap_and_gini_for_batch(model, samples_for_shap, background, nsamples=SHAP_NSAMPLES, seed=seed)
        results[label] = {
            'samples': samples,
            'confidence': predict_confidence(model, samples),
            'entropy': compute_entropy(model, samples),
            'shap_gini': batch_stats['gini'],
            'shap_contrast': batch_stats['contrast'],
            'joint_importance': batch_stats['joint_importance'],
            'normalized_joint_importance': batch_stats['normalized_joint_importance'],
            'preds': batch_stats['preds'],
        }
        print(f'[Negative motion] Processed {label}: {len(samples)} samples, SHAP on {sample_count} samples')

    return results


def plot_real_vs_negative_distributions(negative_results: dict):
    gini_records = []
    confidence_records = []
    contrast_records = []

    for label, stats in negative_results.items():
        contrast_values = np.asarray(stats.get('shap_contrast', np.zeros_like(stats['shap_gini'])), dtype=np.float32)
        for value in stats['shap_gini']:
            gini_records.append({'group': label, 'value': float(value)})
        for value in stats['confidence'][:len(stats['shap_gini'])]:
            confidence_records.append({'group': label, 'value': float(value)})
        for value in contrast_values[:len(stats['shap_gini'])]:
            contrast_records.append({'group': label, 'value': float(value)})

    gini_df = pd.DataFrame(gini_records)
    confidence_df = pd.DataFrame(confidence_records)
    contrast_df = pd.DataFrame(contrast_records)

    _, axes = plt.subplots(1, 3, figsize=(22, 5))
    sns.boxplot(data=gini_df, x='group', y='value', ax=axes[0], hue='group', legend=False)
    sns.stripplot(data=gini_df, x='group', y='value', ax=axes[0], color='black', size=3, alpha=0.35)
    axes[0].set_title('Gini of SHAP: real vs negative motion')
    axes[0].set_xlabel('Input group')
    axes[0].set_ylabel('Gini coefficient')

    sns.boxplot(data=confidence_df, x='group', y='value', ax=axes[1], hue='group', legend=False)
    sns.stripplot(data=confidence_df, x='group', y='value', ax=axes[1], color='black', size=3, alpha=0.35)
    axes[1].set_title('Softmax confidence: real vs negative motion')
    axes[1].set_xlabel('Input group')
    axes[1].set_ylabel('Max softmax probability')

    sns.boxplot(data=contrast_df, x='group', y='value', ax=axes[2], hue='group', legend=False)
    sns.stripplot(data=contrast_df, x='group', y='value', ax=axes[2], color='black', size=3, alpha=0.35)
    axes[2].set_title('SHAP class-contrast: real vs negative motion')
    axes[2].set_xlabel('Input group')
    axes[2].set_ylabel('Pred-vs-runnerup contrast')

    plt.tight_layout()
    plt.show()


def plot_gini_confidence_interaction(negative_results: dict):
    rows = []
    for label, stats in negative_results.items():
        gini_values = np.asarray(stats['shap_gini'], dtype=np.float32)
        confidence_values = np.asarray(stats['confidence'], dtype=np.float32)[:len(gini_values)]
        for gini_value, confidence_value in zip(gini_values, confidence_values):
            rows.append({'group': label, 'gini': float(gini_value), 'confidence': float(confidence_value)})

    df = pd.DataFrame(rows)
    plt.figure(figsize=(8, 6))
    sns.scatterplot(data=df, x='gini', y='confidence', hue='group', alpha=0.75)
    plt.xlabel('Normalized SHAP Gini')
    plt.ylabel('Max softmax confidence')
    plt.title('Confidence vs Gini interaction')
    plt.tight_layout()
    plt.show()


def plot_contrast_confidence_interaction(negative_results: dict):
    rows = []
    for label, stats in negative_results.items():
        contrast_values = np.asarray(stats.get('shap_contrast', np.zeros_like(stats['shap_gini'])), dtype=np.float32)
        confidence_values = np.asarray(stats['confidence'], dtype=np.float32)[:len(contrast_values)]
        for contrast_value, confidence_value in zip(contrast_values, confidence_values):
            rows.append({'group': label, 'contrast': float(contrast_value), 'confidence': float(confidence_value)})

    df = pd.DataFrame(rows)
    plt.figure(figsize=(8, 6))
    sns.scatterplot(data=df, x='contrast', y='confidence', hue='group', alpha=0.75)
    plt.xlabel('SHAP class contrast')
    plt.ylabel('Max softmax confidence')
    plt.title('Confidence vs SHAP class-contrast')
    plt.tight_layout()
    plt.show()


def compare_shap_and_fragility(shap_stability_results: dict, robustness_results: dict):
    """Quantify and visualize whether important joints are also fragile."""
    mean_shap_importance = normalize_importance_vector(shap_stability_results['mean_importance'])
    mean_joint_fragility = normalize_importance_vector(robustness_results['mean_joint_fragility'])

    if len(mean_shap_importance) != len(mean_joint_fragility) or len(mean_shap_importance) == 0:
        print('Skipping SHAP-fragility comparison because the vectors are not aligned.')
        return None

    corr = np.corrcoef(mean_shap_importance, mean_joint_fragility)[0, 1]
    top_shap = np.argsort(-mean_shap_importance)[:5]
    top_fragile = np.argsort(-mean_joint_fragility)[:5]

    print(f'Correlation between normalized SHAP importance and mean fragility: {corr:.4f}')
    print('Top-5 SHAP joints:', top_shap.tolist())
    print('Top-5 fragile joints:', top_fragile.tolist())

    labels = get_joint_labels(len(mean_shap_importance))
    plt.figure(figsize=(8, 6))
    plt.scatter(mean_shap_importance, mean_joint_fragility, c='#4C78A8', alpha=0.75)
    for idx in np.unique(np.concatenate([top_shap[:3], top_fragile[:3]])):
        plt.annotate(labels[idx], (mean_shap_importance[idx], mean_joint_fragility[idx]), textcoords='offset points', xytext=(6, 4), fontsize=9)
    plt.xlabel('Normalized mean SHAP importance')
    plt.ylabel('Normalized mean fragility')
    plt.title('SHAP importance vs fragility')
    plt.tight_layout()
    plt.show()

    return corr


RUN_ROBUSTNESS = True
RUN_NEGATIVE_MOTION = True
STRICT_REAL_NEGATIVE_EVAL = True
REQUIRE_REAL_RANDOM_MOTION = True

if RUN_ROBUSTNESS:
    model = load_model()
    robustness_results = run_robustness_analysis(model, X_test)
    plot_robustness_curve(robustness_results['sigmas'], robustness_results['overall_curve'], robustness_results['overall_std'])
    plot_sample_flip_heatmap(robustness_results['sample_flip_prob'], robustness_results['sigmas'])
    plot_joint_fragility_heatmap(robustness_results['joint_flip_prob'], robustness_results['sigmas'])
    plot_fragility_summary(robustness_results['mean_joint_fragility'])

    # Force challenge-4 linkage in strict mode even if SHAP cell wasn't run earlier.
    if 'shap_stability_results' not in globals():
        print('SHAP stability results not found. Running SHAP stability now for attention-failure linkage...')
        shap_stability_results = run_shap_stability_analysis(
            model=model,
            X_train_values=X_train,
            y_train_values=y_train,
            X_test_values=X_test,
            y_test_values=y_test,
        )
    compare_shap_and_fragility(shap_stability_results, robustness_results)

if RUN_NEGATIVE_MOTION:
    model = load_model()
    if REQUIRE_REAL_RANDOM_MOTION and get_mediapipe_hands_module() is None:
        raise RuntimeError(
            'Strict real-negative mode is enabled, but MediaPipe Hands is unavailable in this kernel. '
            'Use a Python environment with MediaPipe Hands support and restart the kernel. '
            'For non-strict debugging only, set REQUIRE_REAL_RANDOM_MOTION=False.'
        )
    negative_motion_results = run_negative_motion_analysis(
        model=model,
        X_train_values=X_train,
        y_train_values=y_train,
        X_real_eval=X_test,
        strict_real_negative=STRICT_REAL_NEGATIVE_EVAL,
        random_motion_dir=Path('data/random_motion'),
        require_real_random_motion=REQUIRE_REAL_RANDOM_MOTION,
    )
    plot_real_vs_negative_distributions(negative_motion_results)
    plot_gini_confidence_interaction(negative_motion_results)
    plot_contrast_confidence_interaction(negative_motion_results)

### Interpretation note

A steep rise in flip probability at small sigma means the classifier is fragile to noise. For negative motion, real gestures should usually show higher confidence and a different SHAP concentration pattern than random inputs; if Gini overlaps heavily, the intent gate will be weak.

## Gini-Based Intent Gate Evaluation

This section treats higher score as stronger intent. Gini, maximum softmax probability, and negative entropy are all evaluated with the same threshold-sweep and ROC machinery so the comparison is direct.

In [ ]:
# Strict intent gate evaluation with Gini, max-softmax, entropy, SHAP contrast, and fused evidence


def split_gate_indices(y_true: np.ndarray, seed: int = SEED, tune_fraction: float = 0.5):
    """Split gate data into threshold-tuning and final-evaluation partitions."""
    indices = np.arange(len(y_true))
    tune_idx, eval_idx = train_test_split(
        indices,
        test_size=1.0 - tune_fraction,
        random_state=seed,
        stratify=y_true,
    )
    return np.asarray(tune_idx), np.asarray(eval_idx)


def evaluate_binary_metrics(y_true: np.ndarray, y_pred: np.ndarray, score_for_auc: np.ndarray):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    acc = accuracy_score(y_true, y_pred)
    bal_acc = balanced_accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    fpr = fp / max(fp + tn, 1)
    fnr = fn / max(fn + tp, 1)
    specificity = tn / max(tn + fp, 1)

    try:
        auroc = roc_auc_score(y_true, score_for_auc)
        roc_fpr, roc_tpr, roc_thresholds = roc_curve(y_true, score_for_auc)
    except ValueError:
        auroc = float('nan')
        roc_fpr = np.array([0.0, 1.0])
        roc_tpr = np.array([0.0, 1.0])
        roc_thresholds = np.array([np.inf, -np.inf])

    return {
        'accuracy': float(acc),
        'balanced_accuracy': float(bal_acc),
        'precision': float(prec),
        'recall': float(rec),
        'f1': float(f1),
        'mcc': float(mcc),
        'fpr': float(fpr),
        'fnr': float(fnr),
        'specificity': float(specificity),
        'auroc': float(auroc),
        'roc_fpr': roc_fpr,
        'roc_tpr': roc_tpr,
        'roc_thresholds': roc_thresholds,
    }


def fit_gate_threshold_strict(
    y_true: np.ndarray,
    score: np.ndarray,
    higher_is_intent: bool = True,
    n_thresholds: int = 400,
    max_allowed_fpr: float = 0.05,
):
    """Tune threshold with strict false-positive control first, then best balanced accuracy."""
    y_true = np.asarray(y_true).astype(int)
    score = np.asarray(score).astype(float)

    if not higher_is_intent:
        score = -score

    thresholds = np.linspace(score.min(), score.max(), n_thresholds) if score.max() > score.min() else np.array([score.min()])
    records = []

    for threshold in thresholds:
        y_pred = (score >= threshold).astype(int)
        m = evaluate_binary_metrics(y_true, y_pred, score)
        rec = {'threshold': float(threshold), **m}
        records.append(rec)

    feasible = [record for record in records if record['fpr'] <= max_allowed_fpr]
    if feasible:
        best = max(feasible, key=lambda r: (r['balanced_accuracy'], r['f1'], r['recall']))
    else:
        best = max(records, key=lambda r: (r['balanced_accuracy'], -r['fpr'], r['f1']))

    best['all_thresholds'] = records
    best['max_allowed_fpr'] = float(max_allowed_fpr)
    return best


def evaluate_gate_at_threshold(y_true: np.ndarray, score: np.ndarray, threshold: float, higher_is_intent: bool = True):
    """Evaluate one threshold on a holdout set."""
    y_true = np.asarray(y_true).astype(int)
    score = np.asarray(score).astype(float)
    if not higher_is_intent:
        score = -score
    y_pred = (score >= threshold).astype(int)
    return {'threshold': float(threshold), **evaluate_binary_metrics(y_true, y_pred, score)}


def assert_strict_random_motion_inputs(negative_results: dict):
    """Ensure strict challenge-2 evaluation uses true real random-motion negatives."""
    non_real_keys = [key for key in negative_results.keys() if key != 'real']
    if 'real_random_motion' not in non_real_keys:
        raise RuntimeError(
            'Strict evaluation requires real_random_motion negatives. Re-run Cell 11 with REQUIRE_REAL_RANDOM_MOTION=True and working MediaPipe Hands.'
        )
    if 'fallback_random_motion' in non_real_keys:
        raise RuntimeError('Fallback negatives detected. Strict paper evaluation requires true real_random_motion only.')


def build_gate_inputs(negative_results: dict):
    """Align real and negative arrays so every method uses the same sample count."""
    non_real_keys = [key for key in negative_results.keys() if key != 'real']
    available_lengths = [len(negative_results['real']['shap_gini'])] + [len(negative_results[key]['shap_gini']) for key in non_real_keys]
    n = min(available_lengths)

    y_true = np.concatenate([np.ones(n, dtype=int), np.zeros(n * len(non_real_keys), dtype=int)])

    real_conf = negative_results['real']['confidence'][:n]
    real_entropy = negative_results['real']['entropy'][:n]
    real_gini = negative_results['real']['shap_gini'][:n]
    real_contrast = np.asarray(negative_results['real'].get('shap_contrast', np.zeros_like(real_gini)), dtype=np.float32)[:n]

    negative_conf = [negative_results[key]['confidence'][:n] for key in non_real_keys]
    negative_entropy = [negative_results[key]['entropy'][:n] for key in non_real_keys]
    negative_gini = [negative_results[key]['shap_gini'][:n] for key in non_real_keys]
    negative_contrast = [np.asarray(negative_results[key].get('shap_contrast', np.zeros_like(negative_results[key]['shap_gini'])), dtype=np.float32)[:n] for key in non_real_keys]

    scores = {
        'Gini': np.concatenate([real_gini, *negative_gini]),
        'Confidence': np.concatenate([real_conf, *negative_conf]),
        'Entropy (negated)': np.concatenate([-real_entropy, *[-values for values in negative_entropy]]),
        'SHAP Contrast': np.concatenate([real_contrast, *negative_contrast]),
    }
    return y_true, scores


def build_evidence_fusion_inputs(negative_results: dict):
    """Create a feature matrix for the explanation-aware fused gate."""
    non_real_keys = [key for key in negative_results.keys() if key != 'real']
    available_lengths = [len(negative_results['real']['shap_gini'])] + [len(negative_results[key]['shap_gini']) for key in non_real_keys]
    n = min(available_lengths)

    y_true = np.concatenate([np.ones(n, dtype=int), np.zeros(n * len(non_real_keys), dtype=int)])

    def _stack_features(group_name: str) -> np.ndarray:
        stats = negative_results[group_name]
        confidence = np.asarray(stats['confidence'], dtype=np.float32)[:n]
        gini = np.asarray(stats['shap_gini'], dtype=np.float32)[:n]
        contrast = np.asarray(stats.get('shap_contrast', np.zeros_like(gini)), dtype=np.float32)[:n]
        neg_entropy = -np.asarray(stats['entropy'], dtype=np.float32)[:n]
        return np.column_stack([confidence, gini, contrast, neg_entropy])

    real_features = _stack_features('real')
    negative_features = [_stack_features(key) for key in non_real_keys]
    feature_matrix = np.vstack([real_features, *negative_features])
    feature_names = ['confidence', 'gini', 'contrast', 'neg_entropy']
    return y_true, feature_matrix, feature_names


def bootstrap_metric_ci(y_true: np.ndarray, score: np.ndarray, threshold: float, n_boot: int = 250, seed: int = SEED):
    """Bootstrap AUROC and balanced accuracy confidence intervals."""
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true).astype(int)
    score = np.asarray(score).astype(float)
    auc_values = []
    bal_values = []

    for _ in range(n_boot):
        sample_idx = rng.integers(0, len(y_true), size=len(y_true))
        y_sample = y_true[sample_idx]
        score_sample = score[sample_idx]
        if len(np.unique(y_sample)) < 2:
            continue
        try:
            auc_values.append(float(roc_auc_score(y_sample, score_sample)))
        except ValueError:
            pass
        y_pred = (score_sample >= threshold).astype(int)
        bal_values.append(float(balanced_accuracy_score(y_sample, y_pred)))

    def _ci(values):
        if not values:
            return (float('nan'), float('nan'))
        return (float(np.percentile(values, 2.5)), float(np.percentile(values, 97.5)))

    return {
        'auroc_ci': _ci(auc_values),
        'balanced_accuracy_ci': _ci(bal_values),
    }


def paired_permutation_auc_test(y_true: np.ndarray, score_a: np.ndarray, score_b: np.ndarray, n_perm: int = 2000, seed: int = SEED):
    """Paired permutation test for AUROC difference between two methods."""
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true).astype(int)
    score_a = np.asarray(score_a, dtype=np.float64)
    score_b = np.asarray(score_b, dtype=np.float64)

    observed = float(roc_auc_score(y_true, score_a) - roc_auc_score(y_true, score_b))
    count = 0
    for _ in range(n_perm):
        swap_mask = rng.random(len(y_true)) < 0.5
        a_perm = np.where(swap_mask, score_b, score_a)
        b_perm = np.where(swap_mask, score_a, score_b)
        diff = float(roc_auc_score(y_true, a_perm) - roc_auc_score(y_true, b_perm))
        if abs(diff) >= abs(observed):
            count += 1
    p_value = (count + 1.0) / (n_perm + 1.0)
    return observed, float(p_value)


def fit_evidence_fusion_gate(
    y_true: np.ndarray,
    features: np.ndarray,
    feature_names: list[str],
    max_allowed_fpr: float = 0.05,
    split_seed: int = SEED,
):
    """Fit a validation-calibrated logistic gate over confidence, explanation, and entropy evidence."""
    tune_idx, eval_idx = split_gate_indices(y_true, seed=split_seed, tune_fraction=0.5)
    scaler = StandardScaler()
    X_tune = scaler.fit_transform(features[tune_idx])
    X_eval = scaler.transform(features[eval_idx])

    classifier = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=split_seed)
    classifier.fit(X_tune, y_true[tune_idx])

    tune_scores = classifier.predict_proba(X_tune)[:, 1]
    tuned = fit_gate_threshold_strict(
        y_true[tune_idx],
        tune_scores,
        higher_is_intent=True,
        max_allowed_fpr=max_allowed_fpr,
    )

    eval_scores = classifier.predict_proba(X_eval)[:, 1]
    evaluated = evaluate_gate_at_threshold(
        y_true[eval_idx],
        eval_scores,
        tuned['threshold'],
        higher_is_intent=True,
    )

    evaluated['tune_balanced_accuracy'] = tuned['balanced_accuracy']
    evaluated['tune_fpr'] = tuned['fpr']
    evaluated['all_thresholds'] = tuned['all_thresholds']
    evaluated['eval_size'] = int(len(eval_idx))
    evaluated['feature_names'] = feature_names
    evaluated['coefficients'] = {name: float(value) for name, value in zip(feature_names, classifier.coef_[0])}
    evaluated['intercept'] = float(classifier.intercept_[0])
    evaluated['eval_scores'] = eval_scores
    evaluated['y_eval'] = y_true[eval_idx]
    evaluated['bootstrap_ci'] = bootstrap_metric_ci(y_true[eval_idx], eval_scores, tuned['threshold'], n_boot=250, seed=split_seed)
    return evaluated


def run_single_seed_intent_gate_eval(negative_results: dict, max_allowed_fpr: float, split_seed: int):
    """Run one strict split for all gate methods."""
    y_true, scores = build_gate_inputs(negative_results)
    y_fusion, fusion_features, feature_names = build_evidence_fusion_inputs(negative_results)
    if not np.array_equal(y_true, y_fusion):
        raise ValueError('Fusion inputs and baseline gate inputs are not aligned.')

    tune_idx, eval_idx = split_gate_indices(y_true, seed=split_seed, tune_fraction=0.5)
    results_by_method = {}

    for method_name, score in scores.items():
        tuned = fit_gate_threshold_strict(
            y_true[tune_idx],
            score[tune_idx],
            higher_is_intent=True,
            max_allowed_fpr=max_allowed_fpr,
        )
        evaluated = evaluate_gate_at_threshold(
            y_true[eval_idx],
            score[eval_idx],
            tuned['threshold'],
            higher_is_intent=True,
        )
        evaluated['tune_balanced_accuracy'] = tuned['balanced_accuracy']
        evaluated['tune_fpr'] = tuned['fpr']
        evaluated['all_thresholds'] = tuned['all_thresholds']
        evaluated['eval_size'] = int(len(eval_idx))
        evaluated['bootstrap_ci'] = bootstrap_metric_ci(y_true[eval_idx], score[eval_idx], tuned['threshold'], n_boot=250, seed=split_seed)
        evaluated['eval_scores'] = score[eval_idx]
        evaluated['y_eval'] = y_true[eval_idx]
        results_by_method[method_name] = evaluated

    fusion_result = fit_evidence_fusion_gate(
        y_true=y_fusion,
        features=fusion_features,
        feature_names=feature_names,
        max_allowed_fpr=max_allowed_fpr,
        split_seed=split_seed,
    )
    results_by_method['Evidence Fusion'] = fusion_result
    return results_by_method


def summarize_repeated_results(repeated_results: list[dict], method_names: list[str]):
    """Aggregate strict repeated-split metrics for paper-ready reporting."""
    rows = []
    for method_name in method_names:
        auc_values = np.asarray([run[method_name]['auroc'] for run in repeated_results], dtype=np.float64)
        bal_values = np.asarray([run[method_name]['balanced_accuracy'] for run in repeated_results], dtype=np.float64)
        fpr_values = np.asarray([run[method_name]['fpr'] for run in repeated_results], dtype=np.float64)
        f1_values = np.asarray([run[method_name]['f1'] for run in repeated_results], dtype=np.float64)

        rows.append(
            {
                'Method': method_name,
                'AUROC mean': float(auc_values.mean()),
                'AUROC std': float(auc_values.std(ddof=1) if len(auc_values) > 1 else 0.0),
                'AUROC 95% CI': f"[{np.percentile(auc_values, 2.5):.3f}, {np.percentile(auc_values, 97.5):.3f}]",
                'BalAcc mean': float(bal_values.mean()),
                'BalAcc std': float(bal_values.std(ddof=1) if len(bal_values) > 1 else 0.0),
                'BalAcc 95% CI': f"[{np.percentile(bal_values, 2.5):.3f}, {np.percentile(bal_values, 97.5):.3f}]",
                'FPR mean': float(fpr_values.mean()),
                'F1 mean': float(f1_values.mean()),
            }
        )

    table = pd.DataFrame(rows)
    table = table.sort_values(['FPR mean', 'BalAcc mean', 'AUROC mean'], ascending=[True, False, False]).reset_index(drop=True)
    display(table)
    return table


def plot_gate_roc_curves(results_by_method: dict):
    plt.figure(figsize=(8, 7))
    for method_name, result in results_by_method.items():
        plt.plot(result['roc_fpr'], result['roc_tpr'], linewidth=2, label=f"{method_name} (AUC={result['auroc']:.3f})")
    plt.plot([0, 1], [0, 1], linestyle='--', color='gray', linewidth=1)
    plt.xlabel('False positive rate')
    plt.ylabel('True positive rate')
    plt.title('ROC curves for strict intent gate methods (single split)')
    plt.legend()
    plt.tight_layout()
    plt.show()


def plot_gate_ablation_bars(summary_table: pd.DataFrame):
    methods = summary_table['Method'].tolist()
    aucs = summary_table['AUROC mean'].to_numpy()
    bal = summary_table['BalAcc mean'].to_numpy()

    x = np.arange(len(methods))
    width = 0.38
    plt.figure(figsize=(14, 6))
    plt.bar(x - width / 2, aucs, width=width, label='AUROC mean', color='#4C78A8')
    plt.bar(x + width / 2, bal, width=width, label='BalAcc mean', color='#F58518')
    plt.xticks(x, methods, rotation=30, ha='right')
    plt.ylim(0.0, 1.05)
    plt.ylabel('Score')
    plt.title('Strict intent-gate ablation across repeated splits')
    plt.legend()
    plt.tight_layout()
    plt.show()


def run_intent_gate_evaluation(
    negative_results: dict,
    max_allowed_fpr: float = 0.05,
    evaluation_seeds: list[int] | None = None,
):
    """Strict gate evaluation with repeated splits, CIs, and significance testing."""
    assert_strict_random_motion_inputs(negative_results)

    if evaluation_seeds is None:
        evaluation_seeds = list(range(10))

    repeated_results = []
    for split_seed in evaluation_seeds:
        repeated_results.append(
            run_single_seed_intent_gate_eval(
                negative_results=negative_results,
                max_allowed_fpr=max_allowed_fpr,
                split_seed=split_seed,
            )
        )

    reference_results = repeated_results[0]
    plot_gate_roc_curves(reference_results)

    method_names = list(reference_results.keys())
    summary_table = summarize_repeated_results(repeated_results, method_names)
    plot_gate_ablation_bars(summary_table)

    # Statistical test on the representative split: Evidence Fusion vs best non-fusion baseline.
    baseline_candidates = [name for name in method_names if name != 'Evidence Fusion']
    best_baseline = max(baseline_candidates, key=lambda name: reference_results[name]['auroc'])
    observed_diff, p_value = paired_permutation_auc_test(
        reference_results['Evidence Fusion']['y_eval'],
        reference_results['Evidence Fusion']['eval_scores'],
        reference_results[best_baseline]['eval_scores'],
        n_perm=2000,
        seed=SEED,
    )

    fusion_coefficients = reference_results['Evidence Fusion'].get('coefficients', {})
    print('\nEvidence Fusion coefficients (representative split):')
    for name, value in fusion_coefficients.items():
        print(f'- {name}: {value:.4f}')
    print(f"Intercept: {reference_results['Evidence Fusion'].get('intercept', float('nan')):.4f}")
    print(
        f"Permutation test (Evidence Fusion vs {best_baseline}) | "
        f"delta_AUROC={observed_diff:.4f}, p={p_value:.6f}"
    )

    return repeated_results, summary_table


RUN_GATE_EVALUATION = True
STRICT_GATE_SEEDS = list(range(10))
if RUN_GATE_EVALUATION:
    if 'negative_motion_results' not in globals():
        model = load_model()
        negative_motion_results = run_negative_motion_analysis(
            model=model,
            X_train_values=X_train,
            y_train_values=y_train,
            X_real_eval=X_test,
            strict_real_negative=True,
            random_motion_dir=Path('data/random_motion'),
            require_real_random_motion=True,
        )
    gate_runs, gate_table = run_intent_gate_evaluation(
        negative_motion_results,
        max_allowed_fpr=0.05,
        evaluation_seeds=STRICT_GATE_SEEDS,
    )

## Run Order and Strict Evaluation Notes

Suggested strict order:
1. Run setup and asset validation.
2. Ensure real gesture media is under `data/raw/<gesture_class_name>/...`.
3. Ensure real non-intent media is under `data/random_motion/...` (required for strict false-positive evaluation).
4. Run real-data preparation/training to create or reuse `X_landmarks.npy`, `y_labels.npy`, `gesture_model_best.pt`.
5. Run SHAP stability section.
6. Run robustness + negative-motion section (this now forces SHAP-fragility linkage if needed).
7. Run strict intent-gate evaluation section.

Strict evaluation behavior:
- Negative class for intent gating is taken from real random-motion captures (not synthetic perturbations).
- Gate threshold tuning prioritizes low false-positive rate (default FPR <= 0.05) and high balanced accuracy.
- Final reporting includes accuracy, balanced accuracy, F1, MCC, FPR, FNR, specificity, and AUROC.

Performance tip:
- For initial smoke tests, reduce `N_SHAP_EVAL`, `SHAP_NSAMPLES`, or `NEGATIVE_SHAP_SAMPLES`; then restore full settings for final strict results.